<a href="https://colab.research.google.com/github/bfreitas1098/AI-Tuberculosis-Lung-Disease-Detector/blob/main/CAP4630_AI_TB_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1) Cleaning data, dataset splitting, data augmentation

In [8]:
import kagglehub
import cv2
import os
import glob
import numpy as np
from tqdm import tqdm
from tensorflow.keras.preprocessing.image import ImageDataGenerator # the data set
from sklearn.model_selection import train_test_split
path = kagglehub.dataset_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset")
base_folder = os.path.join(path, 'TB_Chest_Radiography_Database')
categories = ['Normal', 'Tuberculosis']

def load_initial_data(base_path, categories, img_size=224):
    X_list = []
    y_list = []
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

#load original images
    for idx, category in enumerate(categories):
        folder_path = os.path.join(base_path, category)
        image_files = glob.glob(os.path.join(folder_path, '*.png'))
        print(f"Loading {category} images...")
        for img_path in tqdm(image_files):
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None: continue

# clean
            img = clahe.apply(img)
            img = cv2.resize(img, (img_size, img_size))
            img = img.astype('float32') / 255.0
            X_list.append(img)
            y_list.append(idx)

# 0 none 1 tb
    return np.array(X_list), np.array(y_list)
X_all, y_all = load_initial_data(base_folder, categories)

#80% test 20% val/test
#train is testing data dont worry about temp
X_train, X_temp, y_train, y_temp = train_test_split(X_all, y_all, test_size=0.20, random_state=42, stratify=y_all)
# 10% Val, 10% Test from temp

#val is data for validation and test is testing data
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# adds new tb images made from the test data (only in Training Set)
normal_train_count = np.sum(y_train == 0)
current_tb_train_count = np.sum(y_train == 1)
needed_tb = normal_train_count - current_tb_train_count
print(f"\nBalancing Training Set: Generating {needed_tb} synthetic Tuberculosis images...")

# setup augmentation (only for the tb)
datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='constant',
    cval=0
)

# isolate training tb images for augmentation
tb_train_indices = np.where(y_train == 1)[0]
tb_train_array = np.expand_dims(X_train[tb_train_indices], axis=-1)

#making the images
augmented_images = []
i = 0
if needed_tb > 0:
    for batch in datagen.flow(tb_train_array, batch_size=1):
        new_img = batch[0].reshape(224, 224)
        augmented_images.append(new_img)
        i += 1
        if i >= needed_tb:
            break

# update X_train and y_train with synthetic data
X_train = np.concatenate([X_train, np.array(augmented_images)])
y_train = np.concatenate([y_train, np.ones(len(augmented_images))])

print(f"\nFinal Dataset Stats:")
print(f"Training - Normal: {np.sum(y_train==0)} | Tuberculosis: {np.sum(y_train==1)}")
print(f"Validation: {len(X_val)} | Test: {len(X_test)}")

Using Colab cache for faster access to the 'tuberculosis-tb-chest-xray-dataset' dataset.
Loading Normal images...


100%|██████████| 3500/3500 [00:59<00:00, 58.73it/s]


Loading Tuberculosis images...


100%|██████████| 700/700 [00:09<00:00, 73.25it/s]



Balancing Training Set: Generating 2240 synthetic Tuberculosis images...

Final Dataset Stats:
Training - Normal: 2800 | Tuberculosis: 2800
Validation: 420 | Test: 420


## 2) Building the CNN Model

This section defines and compiles a Convolutional Neural Network (CNN) model using TensorFlow/Keras. The model is designed to classify chest X-ray images into 'Normal' or 'Tuberculosis'.

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 1)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 222, 222, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,168,513 (42.60 MB)

 Trainable params: 11,168,513 (42.60 MB)

 Non-trainable params: 0 (0.00 B)

Baseline Model: Logistic Regression

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Flatten the image data for the baseline model
# X_train, X_val, X_test are already 224x224, so we flatten them into a 1D array
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Initialize and train the Logistic Regression model
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_flat, y_train)

# Make predictions on the validation set
y_pred_baseline = baseline_model.predict(X_val_flat)

# Evaluate the baseline model
print("Baseline Model (Logistic Regression) Performance on Validation Set:")
print(f"Accuracy: {accuracy_score(y_val, y_pred_baseline):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred_baseline))


Baseline Model (Logistic Regression) Performance on Validation Set:
Accuracy: 0.9571

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.97       350
           1       0.83      0.93      0.88        70

    accuracy                           0.96       420
   macro avg       0.91      0.95      0.93       420
weighted avg       0.96      0.96      0.96       420

